# Модуль 2. Асинхронное программирование: теория и практика

## Введение в модуль

В Модуле 1 мы изучили механизмы Python, лежащие в основе асинхронности: итераторы сохраняют состояние, генераторы приостанавливают выполнение, менеджеры контекста управляют жизненным циклом ресурсов. Теперь мы поднимаемся на уровень выше — к самой **кооперативной многозадачности** и её сердцу, **событийному циклу**.

Этот модуль — самый плотный теоретически. Мы разберём:
- почему **конкурентность** не равна **параллелизму**;
- как устроен **event loop** от системных вызовов до очереди задач;
- что такое **корутина** с точки зрения теории вычислений;
- как управлять множеством корутин и синхронизировать их;
- куда девать **CPU-bound** задачи в однопоточном цикле событий.

## 2.1. Конкурентность, параллелизм, кооперативная многозадачность

### 2.1.1. Concurrency ≠ Parallelism

Это различие фундаментально. Его непонимание — источник большинства ошибок при проектировании асинхронных систем.

**Конкурентность (concurrency)** — это свойство системы обрабатывать **несколько задач в перекрывающиеся периоды времени**. Задачи чередуются: пока одна ждёт, другая выполняется. В один момент реально работает только одна.

**Параллелизм (parallelism)** — это свойство системы **одновременно** выполнять несколько задач на разных вычислительных единицах (ядрах CPU, GPU-стриминг-мультипроцессорах).

**Аналогия:** представьте повара на кухне.

- **Параллелизм** — два повара готовят два блюда одновременно на двух плитах.
- **Конкурентность** — один повар готовит два блюда: поставил воду на плиту, пока она закипает — нарезал овощи, пока овощи тушатся — помешал суп. Блюда готовятся «одновременно» с точки зрения заказчика, но повар в каждый момент делает только одно действие.

Асинхронный Python — это **конкурентность без параллелизма** (если не считать `run_in_executor`). Один поток, один event loop, тысячи корутин, которые по очереди получают управление.

### 2.1.2. Вытесняющая vs кооперативная многозадачность

Операционная система управляет процессами и потоками через **вытесняющую многозадачность (preemptive multitasking)**. Она работает так:

- Каждому потоку выделяется **квант времени** (time slice), например 10 мс.
- По истечении кванта таймер прерывания заставляет CPU переключиться на другой поток.
- Поток не контролирует, когда его прервут. Это решает ОС.

**Кооперативная многозадачность (cooperative multitasking)** работает иначе:

- Задача сама решает, когда уступить управление.
- Она делает это в **точках ожидания**: «я сейчас буду ждать ответа от базы данных, пока жду — займись другими».
- Если задача не уступает управление добровольно, она **блокирует весь цикл**.

В Python `asyncio` — кооперативная модель. Ключевое слово `await` — это именно та точка, где корутина говорит: «я уступаю управление, верните мне его, когда результат будет готов».

### 2.1.3. Почему кооперативная модель эффективна для I/O?

Потому что в веб-приложениях задачи проводят **большую часть времени в ожидании**:

| Операция | Время (порядок) |
|----------|----------------|
| CPU: сложение двух чисел | 1 нс |
| CPU: обращение к L1 кэшу | 1 нс |
| CPU: обращение к оперативной памяти | 100 нс |
| Диск (SSD): чтение 4 КБ | 10 мкс |
| Сеть (внутри дата-центра): RTT | 500 мкс |
| Сеть (межконтинентальная): RTT | 150 мс |
| Человек: клик мышью | 100–300 мс |

Когда сервер ждёт ответа от базы данных (500 мкс–5 мс), CPU мог бы выполнить **миллионы** арифметических операций. Кооперативная многозадачность позволяет не тратить это время впустую.

### 2.1.4. Исторический контекст: от callback-hell к сопрограммам

До появления `async`/`await` в Python 3.5 асинхронный код писался через **колбэки**. Это была эпоха `Twisted` и раннего `Tornado`:

In [ ]:
# Псевдокод в духе Twisted
def handle_request(request):
    d = database.query("SELECT ...")
    d.addCallback(on_data_ready)
    d.addErrback(on_error)

def on_data_ready(result):
    d = cache.set("key", result)
    d.addCallback(on_cache_set)

def on_cache_set(_):
    return Response("OK")

Проблемы:
- **Callback Hell**: вложенность колбэков растёт экспоненциально.
- **Разрыв контекста**: исключения теряются между колбэками.
- **Сложность композиции**: нельзя написать `result1 + result2`, если оба результата приходят асинхронно.

**CSP (Communicating Sequential Processes)** — модель Тони Хоара, где процессы взаимодействуют через каналы. Она легла в основу Go (горутины + channels). Python пошёл другим путём.

**Модель акторов** — каждый «актор» (объект) имеет почтовый ящик и обрабатывает сообщения последовательно. Акторы не разделяют состояние. Это Erlang, Akka. Python не использует эту модель напрямую, но `asyncio` позволяет эмулировать акторов через задачи с очередями.

Python выбрал **сопрограммы (coroutines)** с явными точками уступки (`await`) — золотую середину между читаемостью и производительностью.

### 2.1.5. Математическая подоплека: конечные автоматы и машины состояний

Каждая корутина в `asyncio` — это **конечный автомат** (finite state machine, FSM), хотя и с бесконечным числом состояний из-за произвольных локальных переменных.

Формально, конечный автомат — это кортеж:

$$M = (Q, \Sigma, \delta, q_0, F)$$

где:
- $Q$ — множество состояний,
- $\Sigma$ — алфавит входных символов (событий),
- $\delta: Q \times \Sigma \to Q$ — функция перехода,
- $q_0$ — начальное состояние,
- $F$ — множество финальных состояний.

Для корутины:
- Состояния: `CREATED`, `RUNNING`, `SUSPENDED`, `DONE`, `CANCELLED`.
- События: `await` (приостановка), `resolve future` (возобновление), `cancel` (отмена).
- Переходы: `RUNNING --await--> SUSPENDED --future resolved--> RUNNING --return--> DONE`.

Event loop — это **диспетчер**, который хранит множество таких автоматов и решает, какому из них передать управление в зависимости от внешних событий (готовность сокета, таймаут).

## 2.2. Событийный цикл (Event Loop)

### 2.2.1. Что такое event loop?

Event loop — это **бесконечный цикл**, который:

1. Проверяет, есть ли готовые к выполнению задачи (coroutines).
2. Если есть — передаёт управление следующей задаче.
3. Если нет — **засыпает**, дожидаясь внешнего события (данные пришли по сети, прошёл таймаут).
4. Просыпается, когда событие произошло, и возвращается к шагу 1.

Это **Reactor Pattern** — один из классических паттернов проектирования сетевых приложений.

In [4]:
import asyncio

async def main():
    print("Hello")
    await asyncio.sleep(1)  # <- точка уступки управления
    print("World")

# Запуск event loop (в Jupyter не работает, т.к. он уже запустил event loop для своей работы)
# asyncio.run(main())

# Запуск для Jupyter
await main()

Hello
World


Что происходит под капотом `asyncio.run()`?

1. Создаётся новый event loop.
2. Регистрируется корутина `main()` как задача (Task).
3. Запускается цикл: `while loop.is_running(): loop._run_once()`.
4. Когда `main()` завершится, цикл останавливается и закрывается.

### 2.2.2. Архитектура event loop

Event loop состоит из нескольких структур данных:

| Структура | Назначение |
|-----------|------------|
| **Ready queue** | Очередь корутин, готовых к выполнению прямо сейчас. |
| **Scheduled queue** | Куча (heap) корутин с таймаутами: `call_later`, `sleep`. |
| **Selector** | Механизм ОС для отслеживания готовности файловых дескрипторов (сокетов). |

Алгоритм одной итерации цикла (упрощённо):

In [ ]:
def run_once(self):
    # 1. Переместить просроченные scheduled-задачи в ready queue
    now = self.time()
    while scheduled and scheduled[0].when <= now:
        handle = heapq.heappop(scheduled)
        ready.append(handle)
    
    # 2. Выполнить все задачи из ready queue
    while ready:
        handle = ready.popleft()
        handle._run()
    
    # 3. Определить, сколько можно спать
    timeout = 0 if ready else (scheduled[0].when - now if scheduled else None)
    
    # 4. Дождаться событий от ОС (сокеты, таймауты)
    events = selector.select(timeout)
    
    # 5. Пробудить корутины, ожидающие эти события
    for key, mask in events:
        callback = key.data
        callback()

### 2.2.3. Селекторы: select, poll, epoll, kqueue

Event loop не опрашивает сокеты в цикле («busy waiting») — это сожгло бы CPU. Вместо этого он делегирует ОС через системные вызовы:

**`select`** (BSD, 1983) — самый старый. Передаёт ОС **массивы** файловых дескрипторов. Ограничения:
- Максимальный размер массива: `FD_SETSIZE` = 1024 (на многих системах).
- При каждом вызове ядро копирует массивы из пользовательского пространства в ядро и обратно: $O(n)$.

**`poll`** (System V, 1997) — убирает лимит 1024, но оставляет $O(n)$: ядро проверяет каждый дескриптор в списке.

**`epoll`** (Linux, 2002) — революция. Работает по принципу **готового списка**:
- Регистрация дескриптора: `epoll_ctl(EPOLL_CTL_ADD)` — $O(1)$.
- Ожидание событий: `epoll_wait` — возвращает **только** готовые дескрипторы, $O(k)$, где $k$ — число готовых.
- Не нужно передавать весь массив при каждом вызове.

**`kqueue`** (FreeBSD, macOS) — аналог epoll для BSD-систем.

Python's `asyncio` использует `selectors.DefaultSelector`, который автоматически выбирает лучший доступный механизм (`epoll` на Linux, `kqueue` на macOS, `select` как fallback).

### 2.2.4. Почему `time.sleep(10)` убивает производительность?

Рассмотрим два примера:

In [ ]:
# Вариант А: асинхронный сон
async def handler():
    await asyncio.sleep(10)  # <- уступает управление
    return "done"

# Вариант Б: синхронный сон
async def handler():
    time.sleep(10)  # <- блокирует весь поток!
    return "done"

В варианте А `asyncio.sleep(10)` не засыпает реально. Он регистрирует в event loop: «разбуди меня через 10 секунд» и **уступает управление**. За эти 10 секунд цикл обработает тысячи других запросов.

В варианте Б `time.sleep(10)` — **системный вызов**, который блокирует ОС-поток. Event loop работает в одном потоке. Поток засыпает. Все корутины в цикле замирают. Сервер перестаёт отвечать.

**Это самая распространённая ошибка в асинхронном Python:** вызвать синхронную блокирующую функцию внутри `async def`.

### 2.2.5. API event loop: создание и управление

In [ ]:
import asyncio

# Получить текущий цикл (внутри корутины)
loop = asyncio.get_running_loop()

# Получить или создать цикл (устаревший способ, до Python 3.10)
loop = asyncio.get_event_loop()

# Запустить корутину до завершения
asyncio.run(main())  # предпочтительный способ

# Низкоуровневый запуск
loop = asyncio.new_event_loop()
asyncio.set_event_loop(loop)
try:
    loop.run_until_complete(main())
finally:
    loop.close()

**Важно:** `asyncio.run()` — высокоуровневая функция, предназначенная для точки входа. Она создаёт цикл, запускает корутину, останавливает цикл. Внутри уже запущенного цикла вызывать `asyncio.run()` нельзя — будет ошибка.

### 2.2.6. Математическая подоплека: теория очередей

Event loop — это система массового обслуживания (СМО). Формально:

- **Источник заявок**: входящие HTTP-запросы, таймауты, события сокетов.
- **Очередь**: ready queue + scheduled queue.
- **Сервер**: один «прибор» (одно ядро CPU, один поток), обрабатывающий заявки.
- **Дисциплина обслуживания**: FIFO (First In, First Out) для ready queue.

Это модель **M/M/1** в теории очередей, если считать, что поступление заявок — пуассоновский процесс, а время обслуживания — экспоненциальное распределение.

Для M/M/1 известны формулы:
- Средняя длина очереди: $L_q = \frac{\rho^2}{1-\rho}$, где $\rho = \frac{\lambda}{\mu}$ — коэффициент загрузки.
- Среднее время ожидания: $W_q = \frac{\rho}{\mu(1-\rho)}$.

Когда $\rho \to 1$ (система близка к полной загрузке), очередь и время ожидания стремятся к бесконечности. Это объясняет, почему асинхронный сервер «падает» не линейно, а катастрофически при перегрузке — эффект **лавинной деградации**.

## 2.3. Корутины: `async` / `await`

### 2.3.1. Корутина как объект

Когда вы пишете:

In [5]:
async def fetch_data(url):
    response = await aiohttp.get(url)
    return response.json()

функция `fetch_data` **не выполняется**. Она возвращает объект **корутины**:

In [6]:
coro = fetch_data("https://api.example.com")
print(type(coro))  # <class 'coroutine'>
print(coro)        # <coroutine object fetch_data at 0x...>

<class 'coroutine'>
<coroutine object fetch_data at 0x00000222BF9E01E0>


Корутина — это **объект, хранящий состояние** приостановленной функции: локальные переменные, стек вызовов, точка, на которой остановились. Это прямое развитие генераторов из Модуля 1.

### 2.3.2. Три ключевых объекта: Coroutine, Future, Task

| Объект | Что это | Аналогия |
|--------|---------|----------|
| **Coroutine** | Функция с `async def`, ещё не запущенная. | Черновик задачи. |
| **Future** | Контейнер для результата, который появится в будущем. | Обещание (promise): «когда-нибудь здесь будет значение». |
| **Task** | Корутина, обёрнутая в задачу и поставленная в очередь event loop. | Задача в планировщике. |

**Future** — низкоуровневый примитив. Он имеет состояние и результат:

In [7]:
future = asyncio.Future()
print(future.done())   # False
future.set_result(42)
print(future.result()) # 42

False
42


Обычно вы не создаёте Future вручную — они создаются внутри `asyncio` при await'е на I/O-операциях.

**Task** — это Future + Coroutine. Task запускает корутину в event loop и позволяет отслеживать её выполнение:

In [8]:
async def say_hello():
    await asyncio.sleep(1)
    return "Hello"

# Создание задачи
task = asyncio.create_task(say_hello())
# Теперь say_hello выполняется "в фоне", конкурентно с текущей корутиной

### 2.3.3. Ключевое слово `await`: точка уступки управления

`await` — это **единственная** точка, где корутина может уступить управление event loop. Синтаксически:

In [ ]:
result = await some_awaitable

Что происходит:

1. Проверяется, является ли `some_awaitable` **awaitable** (имеет метод `__await__`).
2. Если объект — корутина, она запускается (или возобновляется).
3. Текущая корутина **приостанавливается** и снимается с выполнения.
4. Event loop получает управление и может запустить другую задачу.
5. Когда `some_awaitable` завершится, текущая корутина **возобновляется** с точки после `await`, и результат присваивается переменной.

**Критически важно:** `await` — это не «ждать, ничего не делая». Это «уступить управление, чтобы другие поработали».

### 2.3.4. Состояния корутины / задачи

In [9]:
import asyncio

async def lifecycle_demo():
    print("RUNNING")
    await asyncio.sleep(0)  # уступаем, но сразу возвращаемся
    print("BACK")

task = asyncio.create_task(lifecycle_demo())

# Состояния, которые может иметь задача:
# PENDING   — создана, но ещё не запущена
# RUNNING   — выполняется прямо сейчас
# DONE      — завершена (успешно или с исключением)
# CANCELLED — была отменена

RUNNING
BACK


Проверка состояний:

In [10]:
print(task.done())       # False, пока не завершится
print(task.cancelled())  # False, если не отменяли
print(task.result())     # Блокирует, пока задача не done; потом возвращает результат

True
False
None


Если задача завершилась с исключением, `task.result()` выбросит это исключение.

### 2.3.5. `await` vs `yield from`

В Python 3.4 корутины создавались через декоратор `@asyncio.coroutine` и `yield from`:

In [ ]:
@asyncio.coroutine
def old_style():
    result = yield from asyncio.sleep(1)
    return result

В Python 3.5+ `async def` и `await` — это **синтаксический сахар** над тем же механизмом, но с важными ограничениями:
- `await` работает только с awaitable-объектами (корутины, задачи, фьючерсы).
- `yield from` работал с любыми итераторами.

Под капотом `await` вызывает `__await__()`, который возвращает итератор. То есть `await` — это `yield from` в специализированной форме.

### 2.3.6. Математическая подоплека: сопрограммы в теории вычислений

В теории языков программирования **сопрограмма (coroutine)** — это обобщение подпрограммы (обычной функции). Различия:

| Подпрограмма | Сопрограмма |
|--------------|-------------|
| Единая точка входа. | Множественные точки входа (возобновление). |
| Выполняется от начала до конца. | Приостанавливается и возобновляется. |
| Завершается с `return`. | Завершается с `return` или `yield`. |
| Вызов подпрограммы — стек растёт. | Сопрограммы могут передавать управление друг другу без стекового вложения. |

Формально, сопрограмма — это **continuation** (продолжение). Continuation — это абстрактное представление «остатка вычисления». Когда корутина встречает `await`, она захватывает свою continuation и передаёт её event loop: «вызови меня снова, когда будут готовы данные».

Это соответствует **Continuation-Passing Style (CPS)** — стилю программирования, где вместо возврата значения функция передаёт результат в callback-continuation. `async`/`await` — это синтаксический сахар над CPS, который делает код читаемым, сохраняя семантику.

## 2.4. Параллельное выполнение корутин

### 2.4.1. `asyncio.gather()`: параллельный запуск

`gather` принимает несколько awaitable и запускает их **конкурентно**:

In [ ]:
import asyncio

async def fetch(url, delay):
    await asyncio.sleep(delay)
    return f"Data from {url}"

async def main():
    results = await asyncio.gather(
        fetch("A", 2),
        fetch("B", 1),
        fetch("C", 3),
    )
    print(results)  # ['Data from A', 'Data from B', 'Data from C']

asyncio.run(main())

Важные детали:
- `gather` возвращает результаты **в том же порядке**, в каком переданы аргументы, независимо от порядка завершения.
- Если одна из корутин выбросит исключение, `gather` по умолчанию **сразу** выбросит её. Остальные задачи продолжат выполняться в фоне, но их результаты будут потеряны.
- Опция `return_exceptions=True` превращает исключения в возвращаемые значения:

In [ ]:
results = await asyncio.gather(
    fetch("A", 1),
    failing_fetch("B"),  # выбросит Exception
    return_exceptions=True
)
# results = ['Data from A', ValueError(...)]

### 2.4.2. `asyncio.wait()`: тонкое управление

`wait` даёт больше контроля, чем `gather`:

In [ ]:
done, pending = await asyncio.wait(
    [task1, task2, task3],
    return_when=asyncio.FIRST_COMPLETED  # или ALL_COMPLETED, FIRST_EXCEPTION
)

| Параметр `return_when` | Когда возвращает управление |
|------------------------|----------------------------|
| `ALL_COMPLETED` | Когда все задачи завершены (по умолчанию). |
| `FIRST_COMPLETED` | Когда хотя бы одна задача завершена. |
| `FIRST_EXCEPTION` | Когда хотя бы одна задача завершилась с исключением. |

In [ ]:
async def main():
    tasks = [
        asyncio.create_task(fetch("A", 2)),
        asyncio.create_task(fetch("B", 1)),
    ]
    
    done, pending = await asyncio.wait(tasks, return_when=asyncio.FIRST_COMPLETED)
    print(f"Готово: {len(done)}, В ожидании: {len(pending)}")
    
    # Не забываем дождаться оставшихся
    if pending:
        await asyncio.wait(pending)

### 2.4.3. `asyncio.as_completed()`: потоковая обработка

Если нужно обрабатывать результаты по мере готовности, не дожидаясь всех:

In [ ]:
async def main():
    coros = [fetch(f"url_{i}", i) for i in [3, 1, 2]]
    
    for coro in asyncio.as_completed(coros):
        result = await coro  # возвращает результат первой готовой, потом второй...
        print(result)
        # Вывод: url_1 (через 1с), url_2 (через 2с), url_3 (через 3с)

`as_completed` возвращает итератор корутин. Каждый `await` на очередной элемент даёт результат следующей завершившейся задачи.

### 2.4.4. `asyncio.create_task()`: фоновые задачи

`create_task` — основной способ запустить корутину «в фоне»:

In [ ]:
async def background_worker():
    while True:
        await process_queue()
        await asyncio.sleep(1)

async def main():
    # Запускаем воркер в фоне
    task = asyncio.create_task(background_worker())
    
    # Одновременно обрабатываем HTTP-запросы
    await start_server()
    
    # При завершении main отменяем фоновую задачу
    task.cancel()
    try:
        await task
    except asyncio.CancelledError:
        print("Фоновая задача отменена")

**Важно:** в Python 3.11+ `asyncio.CancelledError` наследуется от `BaseException`, а не `Exception`. Это сделано для того, чтобы `except Exception` не перехватывал отмену случайно.

### 2.4.5. Отмена задач и таймауты

**Отмена:**

In [ ]:
task = asyncio.create_task(long_operation())

# Отмена
task.cancel()

try:
    await task
except asyncio.CancelledError:
    print("Задача была отменена")

Когда вызывается `task.cancel()`, в корутину **вбрасывается** `CancelledError` в ближайшей точке `await`. Если корутина перехватывает `CancelledError` и не пробрасывает его дальше — отмена будет проигнорирована! Это антипаттерн:

In [ ]:
# ПЛОХО: подавление отмены
async def bad():
    try:
        await asyncio.sleep(10)
    except asyncio.CancelledError:
        print("Отмена? Не слышал")
        # Не пробрасываем! Задача не завершится.

Правильно:

In [ ]:
# ХОРОШО: пробрасываем отмену
async def good():
    try:
        await asyncio.sleep(10)
    except asyncio.CancelledError:
        cleanup()
        raise  # <- обязательно пробрасываем

**Таймауты:**

In [ ]:
# Python 3.11+
async with asyncio.timeout(5):
    await slow_operation()

# До 3.11
await asyncio.wait_for(slow_operation(), timeout=5.0)

`asyncio.timeout` создаёт контекстный менеджер, который отменяет всё внутри блока, если время истекло.

## 2.5. Синхронизация в асинхронном коде

Кооперативная многозадачка даёт иллюзию изоляции, но корутины разделяют **один поток** и **одну память**. Если две корутины одновременно (в перекрывающиеся моменты) модифицируют разделяемую структуру — состояние гонки (race condition) возможно.

### 2.5.1. `asyncio.Lock`: взаимное исключение

In [ ]:
lock = asyncio.Lock()
shared_counter = 0

async def increment():
    global shared_counter
    async with lock:  # <- только одна корутина может войти
        current = shared_counter
        await asyncio.sleep(0)  # имитация "тяжёлой" операции
        shared_counter = current + 1

Без `lock` возможна интерливинг:

| Время | Корутина A | Корутина B |
|-------|-----------|-----------|
| t1 | `current = 0` | |
| t2 | `await sleep(0)` -> уступка | |
| t3 | | `current = 0` |
| t4 | | `shared_counter = 1` |
| t5 | `shared_counter = 1` | |

Результат: два инкремента, но `shared_counter == 1` вместо 2.

`asyncio.Lock` гарантирует, что критическая секция выполняется атомарно относительно других корутин.

### 2.5.2. `asyncio.Semaphore`: ограничение параллелизма

Semaphore — счётчик разрешений. Позволяет одновременно выполнять не более $N$ корутин:

In [ ]:
# Не более 5 одновременных запросов к внешнему API
semaphore = asyncio.Semaphore(5)

async def fetch_limited(url):
    async with semaphore:
        return await aiohttp.get(url)

# Запускаем 100 запросов, но одновременно выполняется только 5
tasks = [asyncio.create_task(fetch_limited(u)) for u in urls]
results = await asyncio.gather(*tasks)

`BoundedSemaphore` отличается тем, что не позволяет освободить больше разрешений, чем было захвачено (защита от багов).

### 2.5.3. `asyncio.Event`, `Condition`, `Barrier`

**Event** — флаг, который могут ждать множество корутин:

In [ ]:
event = asyncio.Event()

async def waiter(name):
    print(f"{name}: жду события...")
    await event.wait()
    print(f"{name}: событие произошло!")

async def setter():
    await asyncio.sleep(2)
    event.set()  # <- все ожидающие пробуждаются

async def main():
    await asyncio.gather(
        waiter("A"), waiter("B"), setter()
    )

**Condition** — Event + Lock. Позволяет ждать условия с атомарной проверкой:

In [ ]:
condition = asyncio.Condition()
queue = []

async def consumer():
    async with condition:
        while not queue:
            await condition.wait()  # отпускает lock, ждёт notify
        item = queue.pop(0)
        print(f"Обработано: {item}")

async def producer():
    async with condition:
        queue.append("data")
        condition.notify_all()  # будим всех ожидающих

**Barrier** — точка синхронизации для $N$ корутин. Все ждут, пока не соберётся нужное число участников, затем все продолжают одновременно.

### 2.5.4. Проблема «голодания» и справедливости

`asyncio.Lock` по умолчанию использует **FIFO-очередь** ожидающих корутин. Это значит, что если корутина A захватила lock, а затем B, C, D встали в очередь — они получат lock в порядке B, C, D.

Но в некоторых реализациях (или при ручном написании lock'ов) возможно **голодание (starvation)**: одна корутина постоянно получает lock, а другие ждут бесконечно.

### 2.5.5. Математическая подоплека: семафоры Дейкстры и deadlock

**Семафор** был изобретён Эдсгером Дейкстрой в 1965 году. Формально:

- Семафор — это целочисленная переменная $S \geq 0$ с двумя атомарными операциями:
  - $P(S)$ (proberen, проверить): если $S > 0$, декрементировать и продолжить; иначе ждать.
  - $V(S)$ (verhogen, увеличить): инкрементировать $S$, возможно пробудить ожидающий процесс.

`asyncio.Lock` — это семафор Дейкстры с $S \in \{0, 1\}$ (бинарный семафор, мьютекс).

**Deadlock** — состояние, когда две или более корутин бесконечно ждут друг друга:

In [ ]:
lock_a = asyncio.Lock()
lock_b = asyncio.Lock()

async def coro_1():
    async with lock_a:
        await asyncio.sleep(0)
        async with lock_b:  # ждёт, пока coro_2 освободит lock_b
            pass

async def coro_2():
    async with lock_b:
        await asyncio.sleep(0)
        async with lock_a:  # ждёт, пока coro_1 освободит lock_a
            pass

Решения:
- Всегда захватывать lock'и в **одинаковом порядке**.
- Использовать `asyncio.timeout` на блокировках.
- Использовать **иерархические lock'и** (не захватывать lock более низкого уровня, удерживая lock высокого).

**Livelock** — корутины не блокированы, но бесконечно «уступают» друг другу, не делая полезной работы. Например, две корутины пытаются захватить lock, видят, что он занят, уступают управление, и так по кругу.

## 2.6. Потоки и процессы в гибридной модели

### 2.6.1. Проблема: CPU-bound задачи блокируют event loop

Асинхронность отлична для I/O. Но что если в корутине нужно выполнить тяжёлое вычисление?

In [ ]:
async def handler():
    # Это заблокирует весь event loop на секунды!
    result = heavy_matrix_computation(data)
    return result

Пока `heavy_matrix_computation` выполняется, event loop не может переключиться на другие задачи. Все запросы встают.

### 2.6.2. `loop.run_in_executor()`: ThreadPoolExecutor

Для **блокирующих, но не CPU-интенсивных** задач (например, синхронная библиотека без async API) используется пул потоков:

In [ ]:
import concurrent.futures

def sync_function(x):
    # Синхронная функция, которую нельзя переписать
    return x ** 2

async def main():
    loop = asyncio.get_running_loop()
    
    # Выполняем в пуле потоков
    with concurrent.futures.ThreadPoolExecutor() as pool:
        result = await loop.run_in_executor(pool, sync_function, 42)
        print(result)

По умолчанию `run_in_executor` использует `ThreadPoolExecutor` с 5 потоками × число CPU.

**Важно:** потоки в Python не дают параллелизма для Python-кода из-за GIL. Но они полезны для:
- Вызовов C-библиотек, которые освобождают GIL (`numpy`, `pandas`, `requests` при ожидании сети).
- Блокирующих системных вызовов.

### 2.6.3. ProcessPoolExecutor: настоящий параллелизм

Для **CPU-bound** задач (ML inference, обработка изображений, криптография) нужны **процессы**:

In [ ]:
import concurrent.futures

def cpu_intensive(n):
    # Тяжёлое вычисление
    return sum(i * i for i in range(n))

async def main():
    loop = asyncio.get_running_loop()
    
    with concurrent.futures.ProcessPoolExecutor() as pool:
        # Задача отправляется в отдельный процесс
        result = await loop.run_in_executor(pool, cpu_intensive, 10_000_000)
        print(result)

Что происходит:
1. `ProcessPoolExecutor` создаёт пул процессов-воркеров.
2. Функция и аргументы **сериализуются** через `pickle` и отправляются воркеру.
3. Воркер выполняет вычисление в отдельном процессе (свой GIL, свои ядра CPU).
4. Результат сериализуется обратно и передаётся в event loop.

**Накладные расходы:**
- Сериализация `pickle` для больших данных (например, тензор изображения) может быть дорогой.
- Создание процесса — сотни миллисекунд. Пул процессов амортизирует это.

### 2.6.4. Паттерн «основной цикл + пул воркеров» для ML

Рекомендуемая архитектура для ML-сервиса на FastAPI:

In [ ]:
┌─────────────────────────────────────┐
│         Event Loop (1 поток)        │
│  ┌─────────┐  ┌─────────┐          │
│  │ Request │  │ Request │  ...      │
│  │ Handler │  │ Handler │          │
│  └────┬────┘  └────┬────┘          │
│       │            │                │
│       └─────┬──────┘                │
│             ▼                       │
│    ┌─────────────────┐              │
│    │ ProcessPool     │              │
│    │ (ML Inference)  │              │
│    │ 4-8 воркеров    │              │
│    └─────────────────┘              │
└─────────────────────────────────────┘

- FastAPI обрабатывает I/O в event loop (чтение запроса, валидация Pydantic, запись ответа).
- Тяжёлый inference отправляется в `ProcessPoolExecutor`.
- Результат возвращается в корутину, которая формирует HTTP-ответ.

In [ ]:
import asyncio
from concurrent.futures import ProcessPoolExecutor
import numpy as np

# Глобальная модель (загружается один раз в каждом процессе-воркере)
_model = None

def init_worker():
    global _model
    _model = load_ml_model()

def predict(features: np.ndarray) -> np.ndarray:
    return _model.predict(features)

# Создаём пул с инициализацией воркеров
_executor = ProcessPoolExecutor(
    max_workers=4,
    initializer=init_worker
)

@app.post("/predict")
async def predict_endpoint(data: InputModel):
    features = np.array(data.features)
    # Отправляем в пул процессов
    loop = asyncio.get_running_loop()
    result = await loop.run_in_executor(_executor, predict, features)
    return {"prediction": result.tolist()}

## 2.7. Асинхронные генераторы и контекстные менеджеры

### 2.7.1. Асинхронные генераторы: `async for`

Если генератор должен `await` внутри себя (например, читать данные из сети порциями), используется **асинхронный генератор**:

In [ ]:
async def fetch_pages(url):
    page = 1
    while True:
        data = await aiohttp.get(f"{url}?page={page}")
        if not data:
            break
        yield data
        page += 1

Использование:

In [ ]:
async for page in fetch_pages("https://api.example.com/items"):
    process(page)

Протокол асинхронного генератора:

| Метод | Назначение |
|-------|------------|
| `__aiter__(self)` | Возвращает асинхронный итератор (обычно `self`). |
| `__anext__(self)` | Асинхронно возвращает следующий элемент. Выбрасывает `StopAsyncIteration`, когда закончилось. |

Под капотом `async for` работает так:

In [ ]:
async_iter = fetch_pages(url).__aiter__()
while True:
    try:
        page = await async_iter.__anext__()
        process(page)
    except StopAsyncIteration:
        break

### 2.7.2. Асинхронные контекстные менеджеры: `async with`

Для ресурсов, у которых инициализация и освобождение асинхронны:

In [ ]:
from contextlib import asynccontextmanager

@asynccontextmanager
async def database_transaction():
    conn = await pool.acquire()
    tx = conn.transaction()
    await tx.start()
    try:
        yield conn  # <- вход в блок with
    except Exception:
        await tx.rollback()
        raise
    else:
        await tx.commit()
    finally:
        await pool.release(conn)

# Использование
async with database_transaction() as conn:
    await conn.execute("INSERT INTO ...")
    # При выходе: commit или rollback + release

Протокол:

| Метод | Назначение |
|-------|------------|
| `__aenter__(self)` | Асинхронная инициализация. Возвращает объект для `as`. |
| `__aexit__(self, exc_type, exc_val, exc_tb)` | Асинхронное освобождение. |

### 2.7.3. Потоковая обработка и backpressure

Асинхронные генераторы часто используются для **потоковой обработки** (streaming). Но здесь кроется опасность: если производитель (generator) генерирует данные быстрее, чем потребитель (consumer) их обрабатывает — происходит **переполнение памяти**.

Это проблема **backpressure** (обратного давления). Решения:

1. **Буферизация с ограничением**: использовать `asyncio.Queue(maxsize=N)`.
2. **Потоковая передача без буферизации**: потребитель запрашивает следующий элемент только когда готов.

Асинхронный генератор естественно реализует второй подход: `__anext__` вызывается только тогда, когда потребитель готов к следующей порции.

In [ ]:
async def producer(queue: asyncio.Queue):
    async for item in fetch_stream():
        await queue.put(item)  # блокируется, если queue полна

async def consumer(queue: asyncio.Queue):
    while True:
        item = await queue.get()
        await process(item)
        queue.task_done()

async def main():
    queue = asyncio.Queue(maxsize=100)  # <- backpressure
    await asyncio.gather(producer(queue), consumer(queue))

## Итог модуля 2

| Концепция | Суть | Ключевой API |
|-----------|------|--------------|
| **Конкурентность** | Перекрывающееся выполнение задач в одном потоке | `async`/`await` |
| **Параллелизм** | Одновременное выполнение на разных ядрах | `ProcessPoolExecutor` |
| **Event Loop** | Цикл, распределяющий управление между корутинами | `asyncio.run()`, `get_running_loop()` |
| **Корутина** | Приостанавливаемая функция с сохранением состояния | `async def` |
| **Task** | Корутина, запущенная в event loop | `asyncio.create_task()` |
| **Future** | Контейнер для будущего результата | `asyncio.Future()` |
| **Gather** | Конкурентный запуск с сохранением порядка результатов | `asyncio.gather()` |
| **Lock** | Взаимное исключение для разделяемых ресурсов | `asyncio.Lock()` |
| **Semaphore** | Ограничение числа одновременных операций | `asyncio.Semaphore()` |
| **Отмена** | Прерывание задачи через `CancelledError` | `task.cancel()` |
| **Таймаут** | Автоматическая отмена по истечении времени | `asyncio.timeout()` |
| **Executor** | Запуск синхронного/CPU-bound кода вне event loop | `run_in_executor()` |
| **Async for/with** | Асинхронные аналоги итераторов и менеджеров | `__aiter__`, `__aenter__` |

В **Модуле 3** мы спустимся на уровень сетевых протоколов: разберём TCP, HTTP, WebSocket — чтобы понимать, какие именно I/O-операции оптимизирует event loop.